# Wound Sensor Patch Classification - Color Features + Machine Learning

**Pipeline:** Image -> Color features -> ML model

Each sensor patch photograph contains several circular colorimetric sensor
spots arranged around a central hub. This notebook:

1. Automatically locates each sensor spot in the image.
2. Computes color statistics (mean RGB / HSV / Lab values and channel ratios)
   inside each spot.
3. Uses those color features (instead of raw pixels) as the input to
   classical machine learning models, matched against documented labels
   (for example `healing` vs `non_healing`) or real measurements.

Two models are built and compared:

- **Model 1**: Random Forest (RF).
- **Model 2**: Gradient Boosting Classifier (a second, different ML model).

The notebook covers: automatic sensor-region detection, color feature
extraction, exploratory analysis and visualization, model training,
evaluation/comparison, and a complete end-to-end prediction function
(read image -> detect regions -> extract features -> predict).


## 1. Configuration

Same two supported dataset layouts as before: folder-per-class, or a single
image folder plus a labels CSV. Update the paths below to match your data.


In [ ]:
import os

# ----------------------------- USER CONFIGURATION -----------------------------
DATA_DIR = "dataset"              # root folder for the folder-per-class layout
CLASS_NAMES = ["healing", "non_healing"]

USE_CSV_LABELS = False            # set True to use the CSV layout instead
IMAGES_DIR = "dataset/images"     # used only when USE_CSV_LABELS = True
LABELS_CSV = "dataset/labels.csv" # used only when USE_CSV_LABELS = True

N_SENSOR_REGIONS = 5               # number of colorimetric spots per patch
RANDOM_SEED = 42
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# --------------------------------------------------------------------------------


## 2. Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from scipy.ndimage import maximum_filter

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import joblib

np.random.seed(RANDOM_SEED)
sns.set_style("whitegrid")


## 3. Automatic sensor-region detection

Each patch shows several colorimetric spots on a textured foam background.
Spots can be brighter or darker than the background depending on the
chemical response, so detection combines two cues:

- **Brightness segmentation** (Otsu threshold on a blurred grayscale image):
  finds spots and connecting arms that are lighter than the background.
- **Local-contrast segmentation** (difference from a heavily blurred
  background estimate): finds spots that are darker than the background too.

The two masks are combined, cleaned up with morphological operations, and
the largest connected component (the sensor star pattern) is kept. The
distance transform of that mask highlights the thickest points, which
correspond to the centers of the circular spots; the `N_SENSOR_REGIONS`
points farthest from the pattern centroid are selected as the sensor spots
(this naturally excludes the central hub) and are then sorted by angle so
that "region 1, 2, 3 ..." refer to consistent physical positions across
images.


In [ ]:
def detect_sensor_regions(bgr_image, n_regions=N_SENSOR_REGIONS,
                            min_frac=0.28, border_frac=0.05, nms_dist=70):
    """
    Detect the circular colorimetric sensor spots in a patch photo.

    Returns
    -------
    regions : list of (x, y, radius) tuples, sorted by angle around the
              pattern centroid (clockwise, starting from the top).
    centroid : (x, y) of the whole sensor pattern.
    mask : binary mask of the detected sensor pattern (for visualization/debugging).
    """
    H, W = bgr_image.shape[:2]
    gray = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2GRAY)

    # Cue 1: brightness-based mask (captures pale spots and connecting arms)
    blurred = cv2.GaussianBlur(gray, (15, 15), 0)
    _, mask_bright = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Cue 2: local-contrast mask (captures spots darker than the background)
    gray_f = gray.astype(np.float32)
    background_estimate = cv2.GaussianBlur(gray_f, (0, 0), sigmaX=45)
    anomaly = np.abs(gray_f - background_estimate)
    anomaly_u8 = np.clip(anomaly * 3, 0, 255).astype(np.uint8)
    _, mask_contrast = cv2.threshold(anomaly_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    mask = cv2.bitwise_or(mask_bright, mask_contrast)

    # Ignore a thin border to avoid photo-edge / vignette artifacts
    margin = int(border_frac * min(H, W))
    border_keep = np.zeros_like(mask)
    border_keep[margin:H - margin, margin:W - margin] = 255
    mask = cv2.bitwise_and(mask, border_keep)

    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((11, 11), np.uint8), borderValue=0)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8), borderValue=0)

    n_comp, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n_comp <= 1:
        raise ValueError("No sensor pattern detected in the image.")
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    mask = np.uint8(labels == largest) * 255
    centroid = np.array(centroids[largest])

    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    is_local_max = (maximum_filter(dist, size=35) == dist) & (dist > min_frac * dist.max())
    ys, xs = np.where(is_local_max)
    candidates = [(x, y, dist[y, x]) for x, y in zip(xs, ys)]
    candidates.sort(key=lambda p: -np.hypot(p[0] - centroid[0], p[1] - centroid[1]))

    picked = []
    for x, y, d in candidates:
        if all((x - fx) ** 2 + (y - fy) ** 2 > nms_dist ** 2 for fx, fy, _ in picked):
            picked.append((x, y, d))
        if len(picked) == n_regions:
            break
    if len(picked) < n_regions:
        raise ValueError(f"Only found {len(picked)} of {n_regions} expected sensor regions.")

    def angle_from_top(p):
        return (np.degrees(np.arctan2(p[0] - centroid[0], -(p[1] - centroid[1]))) + 360) % 360

    picked.sort(key=angle_from_top)
    regions = [(int(x), int(y), max(12, int(0.75 * d))) for x, y, d in picked]
    return regions, centroid, mask


In [ ]:
def visualize_detected_regions(image_path, regions=None):
    bgr = cv2.imread(image_path)
    if regions is None:
        regions, centroid, mask = detect_sensor_regions(bgr)
    vis = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).copy()
    for i, (x, y, r) in enumerate(regions, start=1):
        cv2.circle(vis, (x, y), r, (0, 255, 0), 3)
        cv2.putText(vis, str(i), (x - 8, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
    plt.figure(figsize=(5, 5))
    plt.imshow(vis)
    plt.axis("off")
    plt.title("Detected sensor regions")
    plt.show()
    return regions

# Quick sanity check on one sample image
sample_check_path = None
for cname in CLASS_NAMES:
    candidates = glob.glob(os.path.join(DATA_DIR, cname, "*"))
    if candidates:
        sample_check_path = candidates[0]
        break

if sample_check_path:
    visualize_detected_regions(sample_check_path)
else:
    print("No sample image found yet - set DATA_DIR correctly and re-run this cell.")


## 4. Color feature extraction

For every detected region, the mean color is computed in three color
spaces (RGB, HSV, Lab) plus the relative RGB channel ratios, which helps
normalize out overall lighting differences between photos.


In [ ]:
def extract_color_features(bgr_image, regions):
    """Compute per-region color statistics and return them as a flat dict."""
    features = {}
    hsv_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2HSV)
    lab_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2LAB)

    for i, (x, y, r) in enumerate(regions, start=1):
        roi_mask = np.zeros(bgr_image.shape[:2], dtype=np.uint8)
        cv2.circle(roi_mask, (x, y), r, 255, -1)

        b_mean, g_mean, r_mean = cv2.mean(bgr_image, mask=roi_mask)[:3]
        h_mean, s_mean, v_mean = cv2.mean(hsv_image, mask=roi_mask)[:3]
        l_mean, a_mean, b2_mean = cv2.mean(lab_image, mask=roi_mask)[:3]
        total = r_mean + g_mean + b_mean + 1e-6

        features.update({
            f"region{i}_R": r_mean, f"region{i}_G": g_mean, f"region{i}_B": b_mean,
            f"region{i}_H": h_mean, f"region{i}_S": s_mean, f"region{i}_V": v_mean,
            f"region{i}_L": l_mean, f"region{i}_a": a_mean, f"region{i}_b": b2_mean,
            f"region{i}_R_ratio": r_mean / total,
            f"region{i}_G_ratio": g_mean / total,
            f"region{i}_B_ratio": b_mean / total,
        })
    return features

# Demonstrate the extractor on the sanity-check image
if sample_check_path:
    demo_bgr = cv2.imread(sample_check_path)
    demo_regions, _, _ = detect_sensor_regions(demo_bgr)
    demo_features = extract_color_features(demo_bgr, demo_regions)
    pd.Series(demo_features).to_frame("value")


## 5. Build the feature dataset

In [ ]:
def build_file_label_table():
    records = []
    if USE_CSV_LABELS:
        df_labels = pd.read_csv(LABELS_CSV)
        for _, row in df_labels.iterrows():
            fpath = os.path.join(IMAGES_DIR, row["filename"])
            records.append({"filepath": fpath, "label": row["label"]})
    else:
        for class_name in CLASS_NAMES:
            class_dir = os.path.join(DATA_DIR, class_name)
            if not os.path.isdir(class_dir):
                print(f"Warning: folder not found -> {class_dir}")
                continue
            for fpath in glob.glob(os.path.join(class_dir, "*")):
                if fpath.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    records.append({"filepath": fpath, "label": class_name})
    return pd.DataFrame(records)

file_table = build_file_label_table()
print("Total images found:", len(file_table))
file_table.head()


In [ ]:
rows = []
failed_images = []
for _, row in file_table.iterrows():
    bgr = cv2.imread(row["filepath"])
    if bgr is None:
        failed_images.append(row["filepath"])
        continue
    try:
        regions, centroid, mask = detect_sensor_regions(bgr)
    except ValueError as err:
        failed_images.append(row["filepath"])
        continue
    feats = extract_color_features(bgr, regions)
    feats["label"] = row["label"]
    feats["filepath"] = row["filepath"]
    rows.append(feats)

feature_df = pd.DataFrame(rows)
print("Feature rows built:", len(feature_df))
print("Images skipped (region detection failed):", len(failed_images))
feature_df.head()


In [ ]:
feature_columns = [c for c in feature_df.columns if c not in ("label", "filepath")]
class_names_sorted = sorted(feature_df["label"].unique().tolist())
label_to_idx = {name: i for i, name in enumerate(class_names_sorted)}
idx_to_label = {i: name for name, i in label_to_idx.items()}

X = feature_df[feature_columns].values
y = feature_df["label"].map(label_to_idx).values
print("Feature matrix:", X.shape)
print("Classes:", class_names_sorted)


## 6. Exploratory data analysis and visualization

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=feature_df, x="label", order=class_names_sorted, palette="viridis")
plt.title("Class distribution")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=150)
plt.show()


In [ ]:
# Distribution of a representative feature (mean R channel) per region, split by class
region_r_cols = [c for c in feature_columns if c.endswith("_R")]
melted = feature_df.melt(id_vars="label", value_vars=region_r_cols,
                           var_name="region", value_name="mean_R")

plt.figure(figsize=(10, 5))
sns.boxplot(data=melted, x="region", y="mean_R", hue="label")
plt.title("Mean red channel per sensor region, split by class")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "region_R_by_class.png"), dpi=150)
plt.show()


In [ ]:
# Correlation heatmap across all color features
plt.figure(figsize=(12, 10))
sns.heatmap(feature_df[feature_columns].corr(), cmap="coolwarm", center=0)
plt.title("Correlation between color features")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_correlation.png"), dpi=150)
plt.show()


In [ ]:
# PCA projection of the color feature space, colored by class
scaler_viz = StandardScaler()
X_scaled_viz = scaler_viz.fit_transform(X)
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled_viz)

plt.figure(figsize=(6, 5))
for idx, name in idx_to_label.items():
    mask_i = y == idx
    plt.scatter(X_pca[mask_i, 0], X_pca[mask_i, 1], label=name, alpha=0.7)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of color features")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_projection.png"), dpi=150)
plt.show()


## 7. Train / test split and scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)


## 8. Model 1 - Random Forest

A grid search over a small hyperparameter space selects the best Random
Forest configuration using cross-validation on the training set.


In [ ]:
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 6, 12],
    "min_samples_leaf": [1, 2, 4],
}

rf_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED),
    rf_param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
rf_search.fit(X_train_scaled, y_train)

model1_rf = rf_search.best_estimator_
print("Best Random Forest params:", rf_search.best_params_)
print("Best CV accuracy:", rf_search.best_score_)


In [ ]:
# Feature importance from the Random Forest
importances = pd.Series(model1_rf.feature_importances_, index=feature_columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
top_features.sort_values().plot(kind="barh")
plt.title("Top 15 most important color features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "rf_feature_importance.png"), dpi=150)
plt.show()


## 9. Model 2 - Gradient Boosting Classifier

A second, structurally different ML model trained on the same color
features for comparison.


In [ ]:
model2_gb = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_SEED
)
cv_scores = cross_val_score(model2_gb, X_train_scaled, y_train, cv=5, scoring="accuracy")
print("Gradient Boosting CV accuracy: %.4f (+/- %.4f)" % (cv_scores.mean(), cv_scores.std()))

model2_gb.fit(X_train_scaled, y_train)


In [ ]:
importances_gb = pd.Series(model2_gb.feature_importances_, index=feature_columns)
top_features_gb = importances_gb.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
top_features_gb.sort_values().plot(kind="barh", color="orange")
plt.title("Top 15 most important color features (Gradient Boosting)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "gb_feature_importance.png"), dpi=150)
plt.show()


## 10. Evaluation and model comparison

In [ ]:
def evaluate_model(model, X_eval, y_true, name):
    preds = model.predict(X_eval)
    probs = model.predict_proba(X_eval)

    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, average="weighted", zero_division=0)
    rec = recall_score(y_true, preds, average="weighted", zero_division=0)
    f1 = f1_score(y_true, preds, average="weighted", zero_division=0)

    print(f"--- {name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(classification_report(y_true, preds, target_names=class_names_sorted, zero_division=0))

    cm = confusion_matrix(y_true, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names_sorted, yticklabels=class_names_sorted)
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name.lower().replace(' ', '_')}_confusion_matrix.png"), dpi=150)
    plt.show()

    return {"model": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "probs": probs}

result_rf = evaluate_model(model1_rf, X_test_scaled, y_test, "Model 1 Random Forest")
result_gb = evaluate_model(model2_gb, X_test_scaled, y_test, "Model 2 Gradient Boosting")


In [ ]:
# ROC curves (binary classification case)
if len(class_names_sorted) == 2:
    plt.figure(figsize=(6, 5))
    for result in [result_rf, result_gb]:
        fpr, tpr, _ = roc_curve(y_test, result["probs"][:, 1])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{result['model']} (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve Comparison")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve_comparison.png"), dpi=150)
    plt.show()


In [ ]:
comparison_df = pd.DataFrame([
    {"model": result_rf["model"], "accuracy": result_rf["accuracy"], "precision": result_rf["precision"],
     "recall": result_rf["recall"], "f1": result_rf["f1"]},
    {"model": result_gb["model"], "accuracy": result_gb["accuracy"], "precision": result_gb["precision"],
     "recall": result_gb["recall"], "f1": result_gb["f1"]},
]).set_index("model")

comparison_df.plot(kind="bar", figsize=(8, 5))
plt.title("Model comparison on the test set")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=150)
plt.show()
comparison_df


In [ ]:
# Persist trained models and the scaler
joblib.dump(model1_rf, os.path.join(OUTPUT_DIR, "model1_random_forest.joblib"))
joblib.dump(model2_gb, os.path.join(OUTPUT_DIR, "model2_gradient_boosting.joblib"))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "feature_scaler.joblib"))
print("Models and scaler saved to:", OUTPUT_DIR)


## 11. Full prediction function

This function performs the complete pipeline on a brand new image: read the
image from disk, detect the sensor regions, extract the color features,
scale them the same way as during training, run a chosen trained model, and
return the predicted class with its confidence. It also displays the image
with the detected regions and the prediction.


In [ ]:
def predict_from_image(image_path, model, scaler=scaler, feature_columns=feature_columns,
                          class_names=class_names_sorted, show=True):
    """
    Full inference pipeline for a single sensor patch image.

    Steps:
      1. Read the image from disk.
      2. Detect the sensor regions automatically.
      3. Extract color features from each region.
      4. Scale the features with the training-time scaler.
      5. Run the trained ML model and decode the prediction.

    Parameters
    ----------
    image_path : str
        Path to the image file on disk.
    model : sklearn estimator
        A trained classifier (model1_rf or model2_gb).
    scaler : sklearn StandardScaler
        The scaler fitted on the training features.
    feature_columns : list of str
        Ordered feature names produced by extract_color_features, matching
        the columns the model/scaler were trained on.
    class_names : list of str
        Ordered class names matching the label encoding used at training time.
    show : bool
        If True, displays the image with detected regions and the prediction.

    Returns
    -------
    dict with keys: predicted_label, confidence, class_probabilities, features
    """
    bgr_image = cv2.imread(image_path)
    if bgr_image is None:
        raise FileNotFoundError(f"Could not read image at: {image_path}")

    regions, centroid, mask = detect_sensor_regions(bgr_image)
    features = extract_color_features(bgr_image, regions)

    feature_vector = np.array([[features[col] for col in feature_columns]])
    feature_vector_scaled = scaler.transform(feature_vector)

    predicted_idx = int(model.predict(feature_vector_scaled)[0])
    probabilities = model.predict_proba(feature_vector_scaled)[0]
    predicted_label = class_names[predicted_idx]
    confidence = float(probabilities[predicted_idx])

    if show:
        vis = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB).copy()
        for i, (x, y, r) in enumerate(regions, start=1):
            cv2.circle(vis, (x, y), r, (0, 255, 0), 3)
            cv2.putText(vis, str(i), (x - 8, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
        plt.figure(figsize=(5, 5))
        plt.imshow(vis)
        plt.axis("off")
        plt.title(f"Prediction: {predicted_label} ({confidence * 100:.1f}%)")
        plt.show()

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "class_probabilities": dict(zip(class_names, probabilities.tolist())),
        "features": features,
    }


In [ ]:
# Example usage: replace with the path to any sensor patch image
example_image_path = feature_df["filepath"].iloc[0]

print("Using Model 1 (Random Forest):")
result_example_rf = predict_from_image(example_image_path, model1_rf)
print(result_example_rf["predicted_label"], result_example_rf["confidence"])

print("\nUsing Model 2 (Gradient Boosting):")
result_example_gb = predict_from_image(example_image_path, model2_gb)
print(result_example_gb["predicted_label"], result_example_gb["confidence"])
